In [0]:
# 1. Specify the catalog name
catalog_name = "analytics_dev"

# 2. Manually define schema names and table names
# target_tables = [
#     {"schema": "sales_bronze", "table": "customers"},
#     {"schema": "sales_bronze", "table": "orders"},
#     {"schema": "product", "table": "products"},
#     {"schema": "hr", "table": "employees"},
#     {"schema": "hr", "table": "departments"}
# ]

# 3. Load a CSV file containing schema names and table names
csv_path = "/Workspace/Users/xxx.yyy@zzz.com/target_tables.csv"

target_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .option("encoding", "CP932")
    .csv(csv_path)
)

# 4. Convert the loaded DataFrame into a Python list
target_tables = [
    {
        "schema": row["schema"],
        "table": row["table"]
    }
    for row in target_df.collect()
]

# 5. Get all schemas in the catalog
schemas_df = spark.sql(
    f"SHOW SCHEMAS IN {catalog_name}"
)

# 6. Get all existing tables in each schema
existing_tables_list = []

for row in schemas_df.collect():
    schema_name = row["databaseName"]

    tables_df = spark.sql(
        f"SHOW TABLES IN {catalog_name}.{schema_name}"
    )

    for table_row in tables_df.collect():
        existing_tables_list.append(
            (schema_name, table_row["tableName"])
        )

# 7. Convert the retrieved table list into a Python set
existing_table_set = set(existing_tables_list)

# 8. Check whether each target table exists in the set
results = []

for target in target_tables:
    schema_name = target["schema"]
    table_name = target["table"]

    exists = (schema_name, table_name) in existing_table_set

    results.append({
        "schema": schema_name,
        "table": table_name,
        "exists": exists
    })

# 9. Convert results into a Spark DataFrame
result_df = spark.createDataFrame(results)

display(result_df.select("schema", "table", "exists"))